# Spam Detection
## February 22, 2026

## Load Packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## Load Dataset

In [ ]:
df = pd.read_csv('mail_l7_dataset.csv')
df = df.where(pd.notnull(df), '')

# Encode: spam=0, ham=1
df.loc[df['Category'].str.lower().str.strip() == 'spam', 'Category'] = 0
df.loc[df['Category'].str.lower().str.strip() == 'ham', 'Category'] = 1

print(df.head())

## Features & Target

In [ ]:
X = df['Message'].astype(str)
y = df['Category'].astype(int)

## Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Text to TF-IDF Features

In [ ]:
tfidf = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
X_train_features = tfidf.fit_transform(X_train)
X_test_features = tfidf.transform(X_test)

## Train Models

### Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_features, y_train)
lr_pred = lr.predict(X_test_features)

### Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_features, y_train)
rf_pred = rf.predict(X_test_features.toarray())

### Naive Bayes

In [ ]:
nb = MultinomialNB()
nb.fit(X_train_features, y_train)
nb_pred = nb.predict(X_test_features)

## Evaluate Performance

In [ ]:
def print_metrics(name, y_true, y_pred, pos_label=0):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    print(f"\n{name} Performance:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall   : {rec:.3f}")
    print(f"  F1-Score : {f1:.3f}")

def print_confmat(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    cm_df = pd.DataFrame(
        cm,
        index=['Actual: Ham (1)', 'Actual: Spam (0)'],
        columns=['Pred: Ham (1)', 'Pred: Spam (0)']
    )
    print(f"\n{name} Confusion Matrix:\n{cm_df}")

In [ ]:
for name, pred in [('Logistic Regression', lr_pred), 
                   ('Random Forest', rf_pred), 
                   ('Naive Bayes', nb_pred)]:
    print_metrics(name, y_test, pred, pos_label=0)
    print_confmat(name, y_test, pred)

## Sanity Checks

In [ ]:
check_messages = [
    'Free entry in 2 a weekly competition!',
    'I will meet you at the cafe tomorrow',
    'Congratulations, you won a free ticket'
]

def label_to_str(v):
    return 'Spam (0)' if v == 0 else 'Ham (1)'

print('\n=== SANITY CHECKS ===')
for msg in check_messages:
    msg_tfidf = tfidf.transform([msg])
    
    res_lr = lr.predict(msg_tfidf)[0]
    res_rf = rf.predict(msg_tfidf.toarray())[0]
    res_nb = nb.predict(msg_tfidf)[0]
    
    print(f"\nMessage: {msg}")
    print(f"LR: {label_to_str(res_lr)} | RF: {label_to_str(res_rf)} | NB: {label_to_str(res_nb)}")